# DS-CNN Keyword Spotting — Train, Quantize, Export to TFLite Micro

**SIH26172 — ZeroLatency**

Run cells top to bottom. Upload your recorded keyword samples as a zip when prompted (Cell 3).

Pipeline: Speech Commands (negatives) + your custom keyword (positive) -> MFCC features -> DS-CNN -> int8 quantize -> `.tflite` -> C array for ESP32.

Expected time: ~20-40 min total on Colab's free GPU/CPU runtime.

In [ ]:
# Cell 1: Install deps (Colab already has TF, this just pins/adds extras)
!pip install -q tensorflow==2.15.0 tensorflow-io scipy numpy matplotlib
import tensorflow as tf
print("TF version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

In [ ]:
# Cell 2: Download Google Speech Commands dataset (negatives / "unknown" class + built-in silence)
import os, pathlib, urllib.request, tarfile

DATA_DIR = pathlib.Path('speech_commands')
if not DATA_DIR.exists():
    DATA_DIR.mkdir()
    url = 'https://storage.googleapis.com/download.tensorflow.org/data/speech_commands_v0.02.tar.gz'
    archive = 'speech_commands_v0.02.tar.gz'
    print('Downloading Speech Commands v0.02 (~2.3GB)... this takes a few minutes')
    urllib.request.urlretrieve(url, archive)
    print('Extracting...')
    with tarfile.open(archive) as tar:
        tar.extractall(DATA_DIR)
    os.remove(archive)
    print('Done.')
else:
    print('Already downloaded.')

# We'll use a SUBSET of words as "unknown" negatives (using all ~35 words + all samples
# is overkill and slows training a lot for a hackathon timeline)
UNKNOWN_WORDS = ['yes','no','up','down','left','right','on','off','stop','go',
                 'zero','one','two','three','bed','bird','cat','dog','happy','tree']
print(f'Using {len(UNKNOWN_WORDS)} words as unknown/negative pool')

In [ ]:
# Cell 3: Upload YOUR custom keyword samples
# Zip your recorded folder first: `zip -r keyword.zip data/marvin/` (or whatever your label is)
# Files should be 1-second 16kHz mono WAVs.

from google.colab import files
import zipfile

KEYWORD_LABEL = 'marvin'  # <-- CHANGE THIS to your actual custom keyword

print('Upload your zipped keyword samples now...')
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

KEYWORD_DIR = pathlib.Path('custom_keyword')
KEYWORD_DIR.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(KEYWORD_DIR)

# Find all wavs regardless of nested folder structure from the zip
keyword_wavs = list(KEYWORD_DIR.rglob('*.wav'))
print(f'Found {len(keyword_wavs)} keyword samples')
assert len(keyword_wavs) >= 50, 'Need at least ~50 samples, ideally 150-300. Record more if this is low.'

In [ ]:
# Cell 4: Build file lists for 3 classes: keyword / unknown / silence
import random
random.seed(42)

SC_DIR = DATA_DIR
unknown_wavs = []
for w in UNKNOWN_WORDS:
    folder = SC_DIR / w
    if folder.exists():
        unknown_wavs.extend(list(folder.glob('*.wav')))

random.shuffle(unknown_wavs)
# Cap unknown count to ~4x keyword count so classes aren't wildly imbalanced
n_keyword = len(keyword_wavs)
n_unknown = min(len(unknown_wavs), n_keyword * 4)
unknown_wavs = unknown_wavs[:n_unknown]

# "Silence" class: use Speech Commands' _background_noise_ folder, chopped into 1s clips
bg_dir = SC_DIR / '_background_noise_'
bg_files = list(bg_dir.glob('*.wav')) if bg_dir.exists() else []
print(f'Keyword: {n_keyword} | Unknown: {n_unknown} | Background noise files: {len(bg_files)}')

In [ ]:
# Cell 5: Audio loading + augmentation + MFCC feature extraction
import numpy as np
import tensorflow as tf

SAMPLE_RATE = 16000
CLIP_DURATION_MS = 1000
CLIP_LEN = int(SAMPLE_RATE * CLIP_DURATION_MS / 1000)

# MFCC params (match what you'll compute on-device)
FRAME_LENGTH_MS = 40
FRAME_STEP_MS = 20
N_MFCC = 10
N_MEL_BINS = 40

def load_wav(path, target_len=CLIP_LEN):
    audio_binary = tf.io.read_file(str(path))
    audio, sr = tf.audio.decode_wav(audio_binary, desired_channels=1)
    audio = tf.squeeze(audio, axis=-1)
    audio = audio[:target_len]
    pad = target_len - tf.shape(audio)[0]
    audio = tf.pad(audio, [[0, pad]])
    return audio

def add_background(audio, bg_files, prob=0.6, max_vol=0.15):
    if not bg_files or random.random() > prob:
        return audio
    bg_path = random.choice(bg_files)
    bg_audio = load_wav(bg_path, target_len=CLIP_LEN)
    vol = random.uniform(0.0, max_vol)
    return audio + bg_audio * vol

def time_shift(audio, max_shift=1600):
    shift = random.randint(-max_shift, max_shift)
    return tf.roll(audio, shift, axis=0)

def compute_mfcc(audio):
    stft = tf.signal.stft(
        audio,
        frame_length=int(SAMPLE_RATE * FRAME_LENGTH_MS / 1000),
        frame_step=int(SAMPLE_RATE * FRAME_STEP_MS / 1000),
        fft_length=512,
    )
    spectrogram = tf.abs(stft)
    mel_weight = tf.signal.linear_to_mel_weight_matrix(
        num_mel_bins=N_MEL_BINS,
        num_spectrogram_bins=spectrogram.shape[-1],
        sample_rate=SAMPLE_RATE,
        lower_edge_hertz=20.0,
        upper_edge_hertz=4000.0,
    )
    mel_spectrogram = tf.tensordot(spectrogram, mel_weight, 1)
    log_mel = tf.math.log(mel_spectrogram + 1e-6)
    mfcc = tf.signal.mfccs_from_log_mel_spectrograms(log_mel)[..., :N_MFCC]
    return mfcc  # shape: (time_frames, N_MFCC)

def process_file(path, label, augment=False, bg_files=None):
    audio = load_wav(path)
    if augment:
        audio = time_shift(audio)
        audio = add_background(audio, bg_files or [])
        audio = audio * random.uniform(0.8, 1.2)  # volume jitter
    mfcc = compute_mfcc(audio)
    return mfcc.numpy(), label

# Sanity check shape
test_mfcc, _ = process_file(keyword_wavs[0], 0)
print('MFCC feature shape (time, coeffs):', test_mfcc.shape)
INPUT_SHAPE = test_mfcc.shape

In [ ]:
# Cell 6: Build the full dataset (keyword=0, unknown=1, silence=2)
# Augmentation multiplies keyword samples since we have fewer of those (~150-300 vs thousands of negatives)
#
# NOTE ON SMALL DATASETS (<100 real keyword samples): bumping this multiplier higher
# does NOT substitute for more real recordings -- augmented copies of the same 89
# samples are still fundamentally 89 people/takes, just perturbed. It helps some,
# but the model can still overfit to the underlying 89. Real additional recordings
# from more speakers/takes is the actual fix; this is a partial mitigation while
# more data comes in.

AUGMENT_MULTIPLIER = 6  # increased from 3 -- more augmented variety per real sample helps small datasets
N_SILENCE = n_keyword * 2

X, y = [], []

print('Processing keyword samples (with augmentation)...')
for wav in keyword_wavs:
    for _ in range(AUGMENT_MULTIPLIER):
        mfcc, _ = process_file(wav, 0, augment=True, bg_files=bg_files)
        X.append(mfcc); y.append(0)

print('Processing unknown/negative samples...')
for wav in unknown_wavs:
    mfcc, _ = process_file(wav, 1, augment=True, bg_files=bg_files)
    X.append(mfcc); y.append(1)

print('Generating silence samples from background noise...')
for _ in range(N_SILENCE):
    if bg_files:
        bg_path = random.choice(bg_files)
        bg_audio = load_wav(bg_path, target_len=CLIP_LEN)
        vol = random.uniform(0.0, 0.3)
        audio = bg_audio * vol
    else:
        audio = tf.zeros(CLIP_LEN) + tf.random.normal([CLIP_LEN], stddev=0.005)
    mfcc = compute_mfcc(audio).numpy()
    X.append(mfcc); y.append(2)

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int32)
print('Dataset shape:', X.shape, '| Labels:', np.bincount(y), '(keyword, unknown, silence)')

In [ ]:
# Cell 7: Normalize features + train/val/test split
from sklearn.model_selection import train_test_split

# Normalize using training stats (compute after split to avoid leakage)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

mean = X_train.mean()
std = X_train.std()
X_train = (X_train - mean) / std
X_val = (X_val - mean) / std
X_test = (X_test - mean) / std

print(f'Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}')
print(f'Normalization stats -> mean: {mean:.4f}, std: {std:.4f}  (SAVE THESE for on-device use)')

# Reshape for Conv2D: (batch, time, mfcc, channels)
X_train = X_train[..., np.newaxis]
X_val = X_val[..., np.newaxis]
X_test = X_test[..., np.newaxis]

In [ ]:
# Cell 8: Build the DS-CNN model (compact variant, sized for MCU deployment)
#
# SMALL-DATASET NOTE: with <100 real keyword samples, a full-width DS-CNN memorizes
# the training set almost immediately (this is what the flat val_accuracy line and
# climbing val_loss you saw were showing). This version is intentionally smaller
# (32 filters instead of 64) and adds dropout + L2 weight decay throughout, not just
# at the end -- this trades a bit of ceiling accuracy for much better generalization
# on small data. Once you have 150-300+ real samples, filters can go back up to 64.
from tensorflow.keras import layers, models, regularizers

L2 = regularizers.l2(1e-4)

def build_ds_cnn(input_shape, num_classes=3, filters=32, dropout_rate=0.4):
    inputs = layers.Input(shape=input_shape + (1,))
    x = layers.Conv2D(filters, (10, 4), strides=(2, 2), padding='same',
                       use_bias=False, kernel_regularizer=L2)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.SpatialDropout2D(0.1)(x)

    for _ in range(3):
        # Depthwise
        x = layers.DepthwiseConv2D((3, 3), padding='same', use_bias=False,
                                    depthwise_regularizer=L2)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        # Pointwise
        x = layers.Conv2D(filters, (1, 1), padding='same', use_bias=False,
                           kernel_regularizer=L2)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.SpatialDropout2D(0.1)(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(num_classes, activation='softmax', kernel_regularizer=L2)(x)

    return models.Model(inputs, outputs, name='ds_cnn_kws')

model = build_ds_cnn(INPUT_SHAPE, num_classes=3)
model.compile(optimizer=tf.keras.optimizers.Adam(5e-4),  # lower LR, pairs better with regularization
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.summary()
print(f'\nTotal params: {model.count_params():,} (target: keep well under 100K for MCU comfort)')

In [ ]:
# Cell 9: Train
from sklearn.utils.class_weight import compute_class_weight

# Class weights: penalizes the model more for getting keyword/silence wrong than
# unknown, since unknown is the largest class -- this directly fights the
# "always predict unknown" collapse you saw, independent of augmentation.
class_weights_arr = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = dict(enumerate(class_weights_arr))
print('Class weights:', class_weights, '(higher = model penalized more for missing this class)')

callbacks = [
    # patience raised to 15 -- with regularization, useful learning can be slower
    # to show up in val_accuracy than the previous run's immediate plateau
    tf.keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True, monitor='val_accuracy'),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=6, monitor='val_loss'),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=60,
    batch_size=16,  # smaller batch size for small dataset -- more gradient updates per epoch
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1,
)

# Quick check right after training: did it actually learn to predict keyword at all,
# or collapse again? This catches the failure mode immediately instead of waiting
# for Cell 10's full report.
val_preds = np.argmax(model.predict(X_val, verbose=0), axis=1)
print('\nVal prediction class distribution:', np.bincount(val_preds, minlength=3),
      '(if this is all in one class, collapse is still happening)')

In [ ]:
# Cell 10: Evaluate — this gives you your REAL numbers for the PPT slide
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test accuracy: {test_acc*100:.2f}%')

y_pred = np.argmax(model.predict(X_test), axis=1)
class_names = ['keyword', 'unknown', 'silence']
print('\nClassification report:')
print(classification_report(y_test, y_pred, target_names=class_names))

cm = confusion_matrix(y_test, y_pred)
print('Confusion matrix (rows=true, cols=predicted):')
print(cm)

# False Activation Rate: unknown/silence samples wrongly predicted as keyword
false_activations = ((y_test != 0) & (y_pred == 0)).sum()
n_negative = (y_test != 0).sum()
far = false_activations / n_negative
print(f'\nFalse Activation Rate: {far*100:.3f}% ({false_activations}/{n_negative} negatives misclassified as keyword)')
print('--> To report "false activations per hour": FAR * (3600 / clip_duration_sec) gives rough estimate,')
print('    but the REAL number should come from your streaming/hardware test in Day 6, not this offline test.')

# Recall for keyword specifically (this is your True Positive Rate)
keyword_recall = cm[0][0] / cm[0].sum()
print(f'\nKeyword True Positive Rate (Recall): {keyword_recall*100:.2f}%')

plt.figure(figsize=(5,4))
plt.imshow(cm, cmap='Blues')
plt.xticks(range(3), class_names); plt.yticks(range(3), class_names)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion Matrix')
for i in range(3):
    for j in range(3):
        plt.text(j, i, cm[i][j], ha='center', va='center')
plt.colorbar()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved confusion_matrix.png -- put this straight in your PPT.')

## Live mic test (uncompressed model)

Test the trained float model against your own live voice, right here in Colab, before quantizing/flashing. Uses your laptop/browser mic. Run the cell below multiple times to try several clips -- no need to re-run earlier cells each time.

Requires allowing microphone access in the browser popup when prompted.

In [ ]:
# ============================================================
# LIVE TEST CELL -- paste this as a new cell after Cell 10
# Records ~1.5s from your mic (via browser), tests it against
# the UNCOMPRESSED model (the one still in memory as `model`),
# using the exact same MFCC + normalization pipeline as training.
# Run the cell again to test another clip -- no need to re-run
# earlier cells.
# ============================================================

from IPython.display import Javascript, Audio
from google.colab import output
from base64 import b64decode
import numpy as np
import scipy.io.wavfile as wavfile
import io

RECORD_JS = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time))
var recorder, gumStream;
async function record(time) {
  gumStream = await navigator.mediaDevices.getUserMedia({audio: true});
  recorder = new MediaRecorder(gumStream);
  var chunks = [];
  recorder.ondataavailable = e => chunks.push(e.data);
  recorder.start();
  await sleep(time);
  recorder.stop();
  await new Promise(resolve => recorder.onstop = resolve);
  gumStream.getAudioTracks()[0].stop();
  const blob = new Blob(chunks);
  const buffer = await blob.arrayBuffer();
  const bytes = new Uint8Array(buffer);
  let binary = '';
  for (let i = 0; i < bytes.byteLength; i++) binary += String.fromCharCode(bytes[i]);
  return btoa(binary);
}
"""

def record_from_mic(duration_sec=1.5):
    from google.colab import output as colab_output
    display(Javascript(RECORD_JS))
    b64_audio = colab_output.eval_js(f'record({int(duration_sec*1000)})')
    audio_bytes = b64decode(b64_audio)
    return audio_bytes

def webm_to_wav_array(audio_bytes, target_sr=SAMPLE_RATE):
    # Colab's mic capture comes back as webm/opus -- convert via ffmpeg (preinstalled on Colab)
    import subprocess, tempfile, os
    with tempfile.NamedTemporaryFile(suffix='.webm', delete=False) as f_in:
        f_in.write(audio_bytes)
        in_path = f_in.name
    out_path = in_path.replace('.webm', '.wav')
    subprocess.run(['ffmpeg', '-y', '-i', in_path, '-ar', str(target_sr), '-ac', '1', out_path],
                    capture_output=True, check=True)
    sr, data = wavfile.read(out_path)
    os.remove(in_path); os.remove(out_path)
    # Normalize int16 -> float32 [-1, 1] to match load_wav()'s tf.audio.decode_wav behavior
    data = data.astype(np.float32) / 32768.0
    return data

def classify_live_clip(audio_float, model, mean, std):
    audio_tf = tf.constant(audio_float[:CLIP_LEN], dtype=tf.float32)
    pad = CLIP_LEN - tf.shape(audio_tf)[0]
    audio_tf = tf.pad(audio_tf, [[0, pad]])
    mfcc = compute_mfcc(audio_tf).numpy()
    mfcc_norm = (mfcc - mean) / std
    x = mfcc_norm[np.newaxis, ..., np.newaxis].astype(np.float32)
    probs = model.predict(x, verbose=0)[0]
    class_names = ['keyword', 'unknown', 'silence']
    pred_idx = np.argmax(probs)
    return class_names[pred_idx], probs, class_names

print("Get ready -- recording starts in a moment, say your keyword clearly.")
print("(Run this cell again anytime to test another clip.)\n")

audio_bytes = record_from_mic(duration_sec=1.5)
audio_float = webm_to_wav_array(audio_bytes)

# Play back what was recorded, so you can confirm it actually captured your voice
print("Playback of what was recorded:")
display(Audio(audio_float, rate=SAMPLE_RATE))

pred_class, probs, class_names = classify_live_clip(audio_float, model, mean, std)

print(f"\nPrediction: {pred_class.upper()}")
for name, p in zip(class_names, probs):
    bar = '#' * int(p * 40)
    print(f"  {name:10s} {p*100:5.1f}%  {bar}")


In [ ]:
# Cell 11: Post-training int8 quantization -> TFLite
def representative_dataset():
    for i in range(min(300, len(X_train))):
        yield [X_train[i:i+1].astype(np.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_model = converter.convert()

with open('kws_model_int8.tflite', 'wb') as f:
    f.write(tflite_model)

model_size_kb = len(tflite_model) / 1024
print(f'Quantized model size: {model_size_kb:.1f} KB')
print('This is your "Flash footprint" number for the PPT (RAM usage is measured separately, on ESP32, see firmware notes).')

In [ ]:
# Cell 12: Verify quantized model accuracy didn't collapse (sanity check before flashing)
interpreter = tf.lite.Interpreter(model_path='kws_model_int8.tflite')
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

in_scale, in_zero = input_details[0]['quantization']
out_scale, out_zero = output_details[0]['quantization']
print('Input quant (scale, zero_point):', in_scale, in_zero)
print('Output quant (scale, zero_point):', out_scale, out_zero)
print('SAVE THESE VALUES -- needed in the ESP32 firmware to interpret int8 in/out correctly.\n')

correct = 0
n_eval = min(200, len(X_test))
for i in range(n_eval):
    x = X_test[i:i+1]
    x_int8 = (x / in_scale + in_zero).astype(np.int8)
    interpreter.set_tensor(input_details[0]['index'], x_int8)
    interpreter.invoke()
    out = interpreter.get_tensor(output_details[0]['index'])
    pred = np.argmax(out)
    if pred == y_test[i]:
        correct += 1

print(f'Quantized model accuracy on {n_eval} test samples: {correct/n_eval*100:.2f}%')
print(f'(Compare to float model test accuracy of {test_acc*100:.2f}% -- expect drop of 0-2%)')

In [ ]:
# Cell 13: Convert .tflite to a C array for the ESP32 firmware
def tflite_to_c_array(tflite_path, output_path, var_name='g_kws_model'):
    with open(tflite_path, 'rb') as f:
        data = f.read()
    with open(output_path, 'w') as f:
        f.write(f'// Auto-generated from {tflite_path}\n')
        f.write('#include <cstdint>\n\n')
        f.write(f'alignas(8) const unsigned char {var_name}[] = {{\n')
        for i, b in enumerate(data):
            f.write(f'0x{b:02x}, ')
            if (i + 1) % 12 == 0:
                f.write('\n')
        f.write('\n};\n')
        f.write(f'const int {var_name}_len = {len(data)};\n')
    print(f'Wrote {output_path} ({len(data)} bytes / {len(data)/1024:.1f} KB)')

tflite_to_c_array('kws_model_int8.tflite', 'kws_model_data.h')

# Also save normalization + quantization params the firmware will need
with open('model_params.h', 'w') as f:
    f.write('// Auto-generated params needed by firmware for correct inference\n')
    f.write('#pragma once\n\n')
    f.write(f'#define MFCC_MEAN {mean:.6f}f\n')
    f.write(f'#define MFCC_STD {std:.6f}f\n')
    f.write(f'#define INPUT_SCALE {in_scale:.8f}f\n')
    f.write(f'#define INPUT_ZERO_POINT {in_zero}\n')
    f.write(f'#define OUTPUT_SCALE {out_scale:.8f}f\n')
    f.write(f'#define OUTPUT_ZERO_POINT {out_zero}\n')
    f.write(f'#define INPUT_TIME_FRAMES {INPUT_SHAPE[0]}\n')
    f.write(f'#define INPUT_MFCC_COEFFS {INPUT_SHAPE[1]}\n')
    f.write('// class order: 0=keyword, 1=unknown, 2=silence\n')
print('Wrote model_params.h')

print('\n--- Download these 3 files, you need them for the ESP32 firmware ---')
from google.colab import files as colab_files
colab_files.download('kws_model_data.h')
colab_files.download('model_params.h')
colab_files.download('kws_model_int8.tflite')
colab_files.download('confusion_matrix.png')